In [ ]:
import django

django.setup()

In [ ]:
import logging
import sys

from collections import Counter

import pokebase as pb
from pokebase import APIMetadata, APIResource
from requests.exceptions import HTTPError

from typing import Any, Optional, TypedDict, Unpack

from django.db import transaction
from django.db.models import Model

from pokedex.models import (
    Move,
    Pokemon,
    PokemonAbility,
    PokemonForm,
    PokemonFormType,
    PokemonMove,
    PokemonMoveVersion,
    PokemonSpecies,
    PokemonSpeciesDexEntry,
    PokemonSpeciesVariety,
    PokemonStat,
    PokemonType,
    VersionGameIndex,
    VersionGroup,
)

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
)


logger = logging.getLogger(__name__)


class PokemonBuilder:
    def __get_api_metadata_as_dict(self, api_metadata: APIMetadata) -> dict[str, Any]:
        _ALLOWED_TYPES = (bool, str, int, float, APIMetadata, APIResource)

        if not isinstance(api_metadata, APIMetadata):
            raise Exception(f"objeto informado deve implementar {APIMetadata}")

        result = {}

        for key, value in vars(api_metadata).items():
            if value and not isinstance(value, _ALLOWED_TYPES):
                logger.warning(
                    f"[{key}] value deve ser {_ALLOWED_TYPES} e recebeu {type(value)}"
                )
                continue

            if type(value) == APIMetadata:
                result[key] = self.__get_api_metadata_as_dict(value)
                continue

            result[key] = value

        return result

    def __get_resource_list_names(self, r_list: list) -> list[str]:
        return [obj.name for obj in r_list]

    def __get_object_or_none(self, m: Model, **lkp: dict[str, Any]) -> Optional[Model]:
        try:
            return m.objects.get(**lkp)
        except m.DoesNotExist:
            return None

    def __get_evolves_from_species(self) -> Optional[PokemonSpecies]:
        if not self.resource.species.evolves_from_species:
            return None

        return self.__get_object_or_none(
            PokemonSpecies, name=self.resource.species.evolves_from_species.name
        )

    def __get_pokemon(self) -> tuple[Pokemon, bool]:
        return Pokemon.objects.update_or_create(
            name=self.resource.name,
            defaults={
                "name": self.resource.name,
                "base_experience": self.resource.base_experience,
                "height": self.resource.height,
                "is_default": self.resource.is_default,
                "order": self.resource.order,
                "weight": self.resource.weight,
                "sprites": self.__get_api_metadata_as_dict(self.resource.sprites),
                "cries": self.__get_api_metadata_as_dict(self.resource.cries),
            },
        )

    def __get_pokemonform(self, r: APIResource, **kwargs) -> tuple[PokemonForm, bool]:
        update = kwargs.get("update", False)
        pokemonform_retrieve_method = PokemonForm.objects.get_or_create

        if update:
            pokemonform_retrieve_method = PokemonForm.objects.update_or_create

        logger.debug("linking form '%s' %s" % (r.name, "(U)" if update else ""))
        p_form, created = pokemonform_retrieve_method(
            name=r.name,
            defaults={
                "order": r.order,
                "form_order": r.form_order,
                "is_default": r.is_default,
                "is_battle_only": r.is_battle_only,
                "is_mega": r.is_mega,
                "form_name": r.form_name,
                "sprites": self.__get_api_metadata_as_dict(r.sprites),
                "version_group": self.__get_version_group(str(r.version_group))[0],
            },
        )

        self.__link_pokemon_form_types(p_form, r.types)

        return p_form, created

    def __get_version_group(self, name: str) -> tuple[VersionGroup, bool]:
        version_group = pb.APIResource("version-group", name)

        if not version_group:
            return None, False

        return VersionGroup.objects.get_or_create(
            name=name,
            defaults={
                "generation": version_group.generation.name,
                "order": version_group.order,
                "pokedexes": self.__get_resource_list_names(version_group.pokedexes),
                "regions": self.__get_resource_list_names(version_group.regions),
                "versions": self.__get_resource_list_names(version_group.versions),
            },
        )

    def __link_abilities(self, pokemon: Pokemon):
        for a in self.resource.abilities:
            logger.debug("linking ability: %s" % a.ability)
            pokemon.abilities.add(
                PokemonAbility.objects.update_or_create(
                    slot=a.slot,
                    is_hidden=a.is_hidden,
                    ability=str(a.ability),
                )[0]
            )

    def __link_forms(self, pokemon: Pokemon, **kwargs):
        logger.debug("linking forms for '%s'" % pokemon.name)

        u = kwargs.get("update", False)
        forms = [self.__get_pokemonform(f, update=u)[0] for f in self.resource.forms]

        pokemon.forms.add(*forms)

    def __link_game_indices(self, pokemon: Pokemon):
        for i in self.resource.game_indices:
            logger.debug(
                "linking index %s#%d a %s"
                % (i.version.name, i.game_index, pokemon.name)
            )
            pokemon.game_indices.add(
                VersionGameIndex.objects.update_or_create(
                    game_index=i.game_index, version=i.version.name
                )[0]
            )

    def __link_moves(self, pokemon: Pokemon):
        all_resource_moves = {m.move for m in self.resource.moves}
        all_move_names = {str(m) for m in all_resource_moves}
        existing_moves = Move.objects.filter(name__in=all_move_names)
        e_move_names = set(existing_moves.values_list("name", flat=True))
        m_move_names = {mmn for mmn in all_move_names if mmn not in e_move_names}
        r_vgdetails = {str(m): m.version_group_details for m in self.resource.moves}

        if pokemon.moves.count() == len(all_move_names):
            logger.warning("moves already linked, skipping...")
            return

        logger.debug("%d moves retrieved" % existing_moves.count())
        created_moves = []
        missing_moves = []

        for m in m_move_names:
            r_move = pb.move(m)
            logger.debug("preparing for creating move: %s" % r_move.name)
            missing_moves.append(
                Move(
                    name=r_move.name,
                    accuracy=r_move.accuracy,
                    effect_chance=r_move.effect_chance,
                    pp=r_move.pp,
                    priority=r_move.priority,
                    power=r_move.power,
                    damage_class=r_move.damage_class,
                    generation=r_move.generation,
                    target=r_move.target,
                    type=r_move.type,
                )
            )

        if len(missing_moves):
            created_moves = Move.objects.bulk_create(missing_moves)
            logger.debug("%d moves created" % len(created_moves))

        missing_pmoves = [PokemonMove(move=m) for m in existing_moves]

        if len(missing_pmoves):
            created_pmoves = PokemonMove.objects.bulk_create(missing_pmoves)
            pokemon.moves.add(*created_pmoves)
            logger.debug("%d moves linked" % len(created_pmoves))

        for r_move in all_move_names:
            vg_details = []

            try:
                p_move = pokemon.moves.get(move__name=r_move)
            except:
                print("quebrou em", r_move)
                raise

            for v in r_vgdetails.get(r_move, []):
                vg_details.append(
                    PokemonMoveVersion(
                        version_group=self.__get_version_group(str(v.version_group))[0],
                        move_learn_method=v.move_learn_method,
                        level_learned_at=v.level_learned_at,
                        order=v.order,
                    )
                )

            pm_versions = PokemonMoveVersion.objects.bulk_create(vg_details)
            p_move.version_group_details.add(*pm_versions)

    def __link_pokemon_form_types(self, form: PokemonForm, r_types: APIResource):
        logger.debug("linking '%s' form types" % form.name)
        self.__link_types(PokemonFormType, form, r_types)

    def __link_pokemon_types(self, pokemon: Pokemon):
        logger.debug("linking '%s' types" % pokemon.name)
        self.__link_types(PokemonType, pokemon, self.resource.types)

    def __link_species_pokedex_numbers(self, species: PokemonSpecies):
        r_species = self.resource.species

        for pn in r_species.pokedex_numbers:
            logger.debug("linking pokedex: %s#%d" % (str(pn.pokedex), pn.entry_number))
            species.pokedex_numbers.add(
                PokemonSpeciesDexEntry.objects.update_or_create(
                    entry_number=pn.entry_number,
                    pokedex=str(pn.pokedex),
                )[0]
            )

    def __link_species_varieties(self, species: PokemonSpecies):
        r_species = self.resource.species

        for sv in r_species.varieties:
            logger.debug("linking variety: %s" % str(sv.pokemon))
            sv_pokemon = self.__get_object_or_none(Pokemon, name=str(sv.pokemon))

            if not sv_pokemon and species.name != str(sv.pokemon):
                builder = PokemonBuilder(resource=str(sv.pokemon))
                sv_pokemon = builder.build_pokemon(link_species=True)[0]

            species.varieties.add(
                PokemonSpeciesVariety.objects.update_or_create(
                    is_default=sv.is_default,
                    pokemon=sv_pokemon,
                )[0]
            )

    def __link_species(self, pokemon: Pokemon):
        if pokemon.species:
            logger.debug("%s species already linked!" % pokemon.name)
            return

        r_species = self.resource.species

        logger.debug("linking species: %s" % r_species.name)
        species, _ = PokemonSpecies.objects.update_or_create(
            name=r_species.name,
            defaults={
                "order": r_species.order,
                "gender_rate": r_species.gender_rate,
                "capture_rate": r_species.capture_rate,
                "base_happiness": r_species.base_happiness,
                "is_baby": r_species.is_baby,
                "is_legendary": r_species.is_legendary,
                "is_mythical": r_species.is_mythical,
                "hatch_counter": r_species.hatch_counter,
                "has_gender_differences": r_species.has_gender_differences,
                "forms_switchable": r_species.forms_switchable,
                "growth_rate": r_species.growth_rate,
                "egg_groups": [g.name for g in r_species.egg_groups],
                "color": r_species.color,
                "shape": r_species.shape,
                "evolves_from_species": self.__get_evolves_from_species(),
                "generation": r_species.generation,
            },
        )

        self.__link_species_pokedex_numbers(species)
        self.__link_species_varieties(species)

        pokemon.species = species
        pokemon.save()

    def __link_stats(self, pokemon: Pokemon):
        for s in self.resource.stats:
            logger.debug("linking base %s: %d" % (s.stat, s.base_stat))
            pokemon.stats.add(
                PokemonStat.objects.update_or_create(
                    stat=s.stat,
                    effort=s.effort,
                    base_stat=s.base_stat,
                )[0]
            )

    def __link_types(
        self,
        t_model: PokemonType | PokemonFormType,
        m_obj: Pokemon | PokemonForm,
        r_types: APIResource,
    ):
        for t in r_types:
            logger.debug("linking type: %s" % t.type)
            m_obj.types.add(
                t_model.objects.update_or_create(
                    slot=t.slot,
                    type=t.type,
                )[0]
            )

    @transaction.atomic()
    def build_pokemon(self, **kwargs) -> tuple[Pokemon, bool]:
        logger.info("building Pokemon: %s..." % self.resource.name)
        pokemon, created = self.__get_pokemon()

        if kwargs.get("link_abilities", False):
            logger.info("linking abilities...")
            self.__link_abilities(pokemon)

        if kwargs.get("link_forms", False):
            logger.info("linking forms...")
            self.__link_forms(pokemon, update=kwargs.get("update_form", False))

        if kwargs.get("link_game_indices", False):
            logger.info("linking game_indices...")
            self.__link_game_indices(pokemon)

        if kwargs.get("link_moves", False):
            logger.info("linking moves...")
            self.__link_moves(pokemon)

        if kwargs.get("link_species", False):
            logger.info("linking species...")
            self.__link_species(pokemon)

        if kwargs.get("link_stats", False):
            logger.info("linking stats...")
            self.__link_stats(pokemon)

        if kwargs.get("link_types", False):
            logger.info("linking types...")
            self.__link_pokemon_types(pokemon)

        return pokemon, created

    @transaction.atomic()
    def build_pokemonform(self, **kwargs) -> tuple[PokemonForm, bool]:
        logger.info("building PokemonForm: %s..." % self.resource.name)
        return self.__get_pokemonform(self.resource)

    def __init__(self, resource: str | APIResource = "bulbasaur"):
        if isinstance(resource, APIResource):
            self.resource = resource
            return

        self.resource = pb.pokemon(resource)

In [ ]:
class PaginationParams(TypedDict):
    limit: Optional[int]
    offset: Optional[int]
    link_abilities: bool
    link_game_indices: bool
    link_moves: bool
    link_species: bool
    link_stats: bool
    link_types: bool
    names: list[Any] = []
    update_form: bool = False


def __sync_pokemon(**kwargs: Unpack[PaginationParams]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = kwargs.get(
        "names", [n for n in pb.APIResourceList("pokemon").names][offset:limit]
    )

    for name in names:
        try:
            builder = PokemonBuilder(resource=name)
            builder.build_pokemon(**kwargs)
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                print(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                print(f"Erro HTTP {status_code} no ID {name}, mas continuando...")


def __sync_pokemonform(**kwargs: Unpack[PaginationParams]):
    offset = kwargs.get("offset", None)
    limit = kwargs.get("limit", None)
    names = [n for n in pb.APIResourceList("pokemon-form").names][offset:limit]
    pokemon_names = []

    for index in names:
        try:
            r_form = pb.pokemon_form(index)

            builder = PokemonBuilder(resource=r_form)
            builder.build_pokemonform()

            pokemon_names.append(r_form.pokemon.name)
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                print(f"\n[FIM] Erro HTTP 404 encontrado no ID {index}")
                break
            else:
                print(f"Erro HTTP {status_code} no ID {index}, mas continuando...")

    if offset and limit:
        kwargs.update({"names": pokemon_names})

    __sync_pokemon(**kwargs)

In [ ]:
__sync_pokemonform(
    offset=905,
    link_abilities=True,
    link_forms=True,
    link_game_indices=True,
    link_species=True,
    link_stats=True,
    link_types=True,
)